# The Same From The Command Line

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

Everything the package does is reachable from Python, and the package also
installs a command, `opengate-gate-tree`, that does the most common part of it
without writing any: read one tree from a GATE file, keep the branches asked
for, write them in the format asked for, and optionally write a report beside
them.

This notebook runs that command on the example files, option by option, and
then does the same run in Python to show that the two give the same file. The
command knows nothing the library does not; it is the better tool when files
are processed in bulk from a shell, or when whatever comes next is not
Python.

## What will you learn from this notebook

- how to extract a tree, or some of its branches, from the command line
- how the output file is named, and where the report goes
- how to tell from the exit code whether a run succeeded
- how to deal with a file whose hits are split over several trees
- when to turn the structure check off, and what that costs
- how the same run looks in Python, and when the command is the handier of the
  two

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

The command is run from the notebook with IPython's `!`, which hands the rest
of the line to the shell. In a terminal, type the same line without the `!`.
`{name}` in such a line is replaced by the value of the Python variable `name`
before the shell sees it.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The paths are handed to the command relative to the directory the notebook
runs in, so that what the command prints is short and the same on every
machine. The files are described in [`../data/README.md`](../data/README.md).

In [2]:
import os
import sys


def relative(path):
    """The path as seen from the directory the notebook runs in."""
    return os.path.relpath(path)


output = relative(paths.output_dir("tutorial-04-command-line"))
scanner = relative(paths.input_example("hits-a2-cylindrical-pet.root"))
per_run = relative(paths.input_example("hits-one-tree-per-run.root"))
per_detector = relative(paths.input_example("hits-one-tree-per-detector.root"))

output, scanner

('.tmp/tutorial-04-command-line',
 '../data/input-examples/hits-a2-cylindrical-pet.root')

The command is installed beside the Python interpreter running this
notebook. A terminal in which the environment is activated finds it on its own;
the kernel of a notebook may not have that directory on its `PATH`, so it is
added here. `NO_COLOR` keeps the help text free of terminal colour codes, which
a saved notebook would show as noise:

In [3]:
os.environ["PATH"] = os.pathsep.join([os.path.dirname(sys.executable), os.environ["PATH"]])
os.environ["NO_COLOR"] = "1"

### What The Command Takes

`--help` lists every option. Five of them are needed for every run: the input
file, the output directory, the title of the output file, the tree, and the
format. The help lists the allowed trees and formats by the names of their enum
members, `GateTree.HITS` and so on; on the command line they are written as
`Hits` and `csv`, as in every example below.

In [4]:
!opengate-gate-tree --help

usage: opengate-gate-tree [-h] [--input-gate-root-file INPUT_GATE_ROOT_FILE]
                          [--output-dir OUTPUT_DIR]
                          [--output-file-title OUTPUT_FILE_TITLE]
                          [--gate-tree {GateTree.HITS,GateTree.SINGLES,GateTree.COINCIDENCES}]
                          [--output-file-format {OutputFileFormat.ROOT,OutputFileFormat.HDF5,OutputFileFormat.CSV}]
                          [--branches-to-extract BRANCHES_TO_EXTRACT [BRANCHES_TO_EXTRACT ...]]
                          [--input-tree-name INPUT_TREE_NAME]
                          [--merge-hits-trees] [--statistics]
                          [--skip-hits-validation]

options:
  -h, --help            show this help message and exit
  --input-gate-root-file INPUT_GATE_ROOT_FILE
                        Path to the gate '*.root' file to process
  --output-dir OUTPUT_DIR
                        Path to the directory where the output files will be
                        saved
  --output-f

### Extracting A Tree

The whole "Hits" tree of the scanner file, written as CSV:

In [5]:
!opengate-gate-tree \
    --input-gate-root-file {scanner} \
    --output-dir {output} \
    --output-file-title scanner \
    --gate-tree Hits \
    --output-file-format csv

2026-10-12 03:23:45,767 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-a2-cylindrical-pet.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='scanner', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=[], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)


2026-10-12 03:23:45,827 [INFO] opengate_gate_tree: Hits tree variant: System (A2), stored as 'Hits', system cylindricalPET / OPET, 46 branches
2026-10-12 03:23:45,827 [INFO] opengate_gate_tree: Extracted 500 entries and 46 branches from tree 'Hits'.


2026-10-12 03:23:45,844 [INFO] opengate_gate_tree: Done. Output saved to file: .tmp/tutorial-04-command-line/scanner.hits.csv


The log says what was done: the configuration of the run, the structure the
tree was recognised as — a `cylindricalPET` system, 46 branches — how much was
extracted, and where it went.

The output is named `<title>.<tree>.<format>`. The tree is part of the name
because one GATE file holds several trees, and extracting two of them under one
title should not write both to the same file:

In [6]:
sorted(os.listdir(output))

['all-runs.hits.csv',
 'batch',
 'from-python',
 'run-1.hits.csv',
 'runs.hits.csv',
 'scanner-energies.hits.hdf5',
 'scanner-energies.hits.stats.json',
 'scanner-selection.hits.root',
 'scanner.hits.csv',
 'selection-again.hits.csv']

### Keeping Some Branches, And A Report

`--branches-to-extract` takes the branches to keep, separated by spaces, in the
order they should have. `--statistics` writes a report next to the data and
prints it to the log. This time the output is HDF5:

In [7]:
!opengate-gate-tree \
    --input-gate-root-file {scanner} \
    --output-dir {output} \
    --output-file-title scanner-energies \
    --gate-tree Hits \
    --output-file-format hdf5 \
    --branches-to-extract runID eventID trackID edep processName \
    --statistics

2026-10-12 03:23:46,690 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-a2-cylindrical-pet.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='scanner-energies', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.HDF5: 'hdf5'>, branches_to_extract=['runID', 'eventID', 'trackID', 'edep', 'processName'], input_tree_name=None, merge_hits_trees=False, write_statistics=True, skip_hits_validation=False)
2026-10-12 03:23:46,710 [INFO] opengate_gate_tree: Hits tree variant: System (A2), stored as 'Hits', system cylindricalPET / OPET, 46 branches
2026-10-12 03:23:46,710 [INFO] opengate_gate_tree: Extracted 500 entries and 5 branches from tree 'Hits'.
2026-10-12 03:23:46,715 [INFO] opengate_gate_tree: Tree: Hits
Hits tree variant: System (A2), stored as 'Hits', system cylindricalPET / OPET, 46 branches
Entries: 500
Events: 149 (counted by runID, eventID)
Runs: 1
Tracks: 259 (cou

149 events, counted by the pair `(runID, eventID)` since both were
extracted, and 120 MeV deposited in all. The report sits beside the data under
the same title, as JSON, so that the next step can read it as easily as a
person. Tutorial 07 is about what it holds.

In [8]:
sorted(os.listdir(output))

['all-runs.hits.csv',
 'batch',
 'from-python',
 'run-1.hits.csv',
 'runs.hits.csv',
 'scanner-energies.hits.hdf5',
 'scanner-energies.hits.stats.json',
 'scanner-selection.hits.root',
 'scanner.hits.csv',
 'selection-again.hits.csv']

### Did It Work?

A script running the command needs an answer it can test, and the exit code is
that answer: `0` when the output was written, `1` when the run failed — the
reason is on the log — and `2` when the arguments could not be parsed.

A branch the tree does not hold fails the run, and nothing is written:

In [9]:
!opengate-gate-tree \
    --input-gate-root-file {scanner} \
    --output-dir {output} \
    --output-file-title scanner-typo \
    --gate-tree Hits \
    --output-file-format csv \
    --branches-to-extract eventID energy; \
echo "exit code: $?"

2026-10-12 03:23:47,845 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-a2-cylindrical-pet.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='scanner-typo', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=['eventID', 'energy'], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)


2026-10-12 03:23:47,861 [ERROR] opengate_gate_tree: An error occurred: Branches not found in the tree: ['energy']. Branches available in the tree: ['PDGEncoding', 'trackID', 'parentID', 'trackLocalTime', 'time', 'edep', 'stepLength', 'trackLength', 'posX', 'posY', 'posZ', 'localPosX', 'localPosY', 'localPosZ', 'sourcePosX', 'sourcePosY', 'sourcePosZ', 'sourceID', 'eventID', 'runID', 'volumeID', 'processName', 'gantryID', 'rsectorID', 'moduleID', 'submoduleID', 'crystalID', 'layerID', 'momDirX', 'momDirY', 'momDirZ', 'photonID', 'nPhantomCompton', 'nCrystalCompton', 'nPhantomRayleigh', 'nCrystalRayleigh', 'nInteractions', 'primaryID', 'axialPos', 'rotationAngle', 'comptVolName', 'RayleighVolName', 'sourceType', 'decayType', 'gammaType', 'decayIndex'].


exit code: 1


A needed option left out fails the run the same way, before the input file
is opened:

In [10]:
!opengate-gate-tree --input-gate-root-file {scanner} --output-dir {output}; echo "exit code: $?"

2026-10-12 03:23:48,826 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-a2-cylindrical-pet.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title=None, gate_tree=None, output_file_format=None, branches_to_extract=[], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:48,826 [ERROR] opengate_gate_tree: An error occurred: Output file title is required.


exit code: 1


`2` is kept for arguments that cannot be parsed at all, such as a format the
command does not know. argparse prints the usage, then the reason:

In [11]:
!opengate-gate-tree \
    --input-gate-root-file {scanner} \
    --output-dir {output} \
    --output-file-title scanner-xlsx \
    --gate-tree Hits \
    --output-file-format xlsx; \
echo "exit code: $?"

usage: opengate-gate-tree [-h] [--input-gate-root-file INPUT_GATE_ROOT_FILE]
                          [--output-dir OUTPUT_DIR]
                          [--output-file-title OUTPUT_FILE_TITLE]
                          [--gate-tree {GateTree.HITS,GateTree.SINGLES,GateTree.COINCIDENCES}]
                          [--output-file-format {OutputFileFormat.ROOT,OutputFileFormat.HDF5,OutputFileFormat.CSV}]
                          [--branches-to-extract BRANCHES_TO_EXTRACT [BRANCHES_TO_EXTRACT ...]]
                          [--input-tree-name INPUT_TREE_NAME]
                          [--merge-hits-trees] [--statistics]
                          [--skip-hits-validation]
opengate-gate-tree: error: argument --output-file-format: invalid parse_output_file_format value: 'xlsx'


exit code: 2


The reason names the function that refused the value,
`parse_output_file_format`, rather than the option's allowed values; those are
in `--help`.

None of the failed runs wrote anything:

In [12]:
[name for name in os.listdir(output) if "typo" in name or "xlsx" in name]

[]

### Hits Split Over Several Trees

GATE writes one tree per run when a simulation has several runs. This file has
three, `Hits`, `Hits_run1` and `Hits_run2`. Asked for hits without saying
which, the command reads the one named `Hits` and warns that two more are
there:

In [13]:
!opengate-gate-tree \
    --input-gate-root-file {per_run} \
    --output-dir {output} \
    --output-file-title runs \
    --gate-tree Hits \
    --output-file-format csv \
    --branches-to-extract runID eventID edep

2026-10-12 03:23:50,761 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-one-tree-per-run.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='runs', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=['runID', 'eventID', 'edep'], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:50,792 [WARNING] opengate_gate_tree: File ../data/input-examples/hits-one-tree-per-run.root holds hits in 2 more tree(s) besides 'Hits': ['Hits_run1', 'Hits_run2']. Only 'Hits' is read. Name one of them to read it instead, or read them all together.
2026-10-12 03:23:50,799 [INFO] opengate_gate_tree: Hits tree variant: No system (A1), stored as 'Hits', 40 branches
2026-10-12 03:23:50,799 [INFO] opengate_gate_tree: Extracted 500 entries and 3 branches from tree 'Hits'.
2026-10-12 03:23:50,803 [INFO] o

`--input-tree-name` reads another one of them:

In [14]:
!opengate-gate-tree \
    --input-gate-root-file {per_run} \
    --output-dir {output} \
    --output-file-title run-1 \
    --gate-tree Hits \
    --output-file-format csv \
    --branches-to-extract runID eventID edep \
    --input-tree-name Hits_run1

2026-10-12 03:23:51,630 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-one-tree-per-run.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='run-1', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=['runID', 'eventID', 'edep'], input_tree_name='Hits_run1', merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:51,651 [INFO] opengate_gate_tree: Hits tree variant: No system (A1), stored as 'Hits_run1', 40 branches
2026-10-12 03:23:51,651 [INFO] opengate_gate_tree: Extracted 500 entries and 3 branches from tree 'Hits'.
2026-10-12 03:23:51,654 [INFO] opengate_gate_tree: Done. Output saved to file: .tmp/tutorial-04-command-line/run-1.hits.csv


Read the log carefully here: "from tree 'Hits'" names the kind of tree that
was extracted, not the tree in the file. The line above it says which one that
was — `stored as 'Hits_run1'`.

`--merge-hits-trees` reads all three as one dataset, 1500 entries, with a
column `sourceTreeName` saying which tree each row came from:

In [15]:
!opengate-gate-tree \
    --input-gate-root-file {per_run} \
    --output-dir {output} \
    --output-file-title all-runs \
    --gate-tree Hits \
    --output-file-format csv \
    --branches-to-extract runID eventID edep \
    --merge-hits-trees

2026-10-12 03:23:52,443 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-one-tree-per-run.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='all-runs', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=['runID', 'eventID', 'edep'], input_tree_name=None, merge_hits_trees=True, write_statistics=False, skip_hits_validation=False)


2026-10-12 03:23:52,473 [INFO] opengate_gate_tree: Hits tree variant: No system (A1), stored as 'Hits', 40 branches
2026-10-12 03:23:52,473 [INFO] opengate_gate_tree: Extracted 1500 entries and 4 branches from tree 'Hits'.
2026-10-12 03:23:52,478 [INFO] opengate_gate_tree: Done. Output saved to file: .tmp/tutorial-04-command-line/all-runs.hits.csv


In [16]:
import pandas as pd

all_runs = pd.read_csv(os.path.join(output, "all-runs.hits.csv"))
all_runs.groupby("sourceTreeName")["runID"].unique()

sourceTreeName
Hits         [0]
Hits_run1    [1]
Hits_run2    [2]
Name: runID, dtype: object

A file that writes one tree per sensitive detector is a harder case: none of
its trees is called `Hits`, so there is no obvious one to read, and the command
refuses to guess — and says which two options would settle it:

In [17]:
!opengate-gate-tree \
    --input-gate-root-file {per_detector} \
    --output-dir {output} \
    --output-file-title detectors \
    --gate-tree Hits \
    --output-file-format csv; \
echo "exit code: $?"

2026-10-12 03:23:53,441 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-one-tree-per-detector.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='detectors', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=[], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:53,462 [ERROR] opengate_gate_tree: An error occurred: File ../data/input-examples/hits-one-tree-per-detector.root holds hits in 2 trees: ['Hits_DET_INNER', 'Hits_DET_OUTER']. None of them is named 'Hits', so the one to read cannot be chosen. Name the tree to read, or read them all together as one dataset.
2026-10-12 03:23:53,463 [ERROR] opengate_gate_tree: Use --input-tree-name to read one of them, or --merge-hits-trees to read them all.


exit code: 1


### Turning The Structure Check Off

Before reading a "Hits" tree, the command recognises which structure it has
and checks the tree against it. A file the package does not recognise stops the
run. One such file is close at hand: a ROOT file the command wrote itself from
a selection of branches, which is no longer a whole structure.

In [18]:
!opengate-gate-tree \
    --input-gate-root-file {scanner} \
    --output-dir {output} \
    --output-file-title scanner-selection \
    --gate-tree Hits \
    --output-file-format root \
    --branches-to-extract eventID edep posX posY posZ

2026-10-12 03:23:54,227 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('../data/input-examples/hits-a2-cylindrical-pet.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='scanner-selection', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.ROOT: 'root'>, branches_to_extract=['eventID', 'edep', 'posX', 'posY', 'posZ'], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:54,247 [INFO] opengate_gate_tree: Hits tree variant: System (A2), stored as 'Hits', system cylindricalPET / OPET, 46 branches
2026-10-12 03:23:54,247 [INFO] opengate_gate_tree: Extracted 500 entries and 5 branches from tree 'Hits'.
2026-10-12 03:23:54,250 [INFO] opengate_gate_tree: Done. Output saved to file: .tmp/tutorial-04-command-line/scanner-selection.hits.root


In [19]:
selection = os.path.join(output, "scanner-selection.hits.root")

!opengate-gate-tree \
    --input-gate-root-file {selection} \
    --output-dir {output} \
    --output-file-title selection-again \
    --gate-tree Hits \
    --output-file-format csv; \
echo "exit code: $?"

2026-10-12 03:23:55,019 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('.tmp/tutorial-04-command-line/scanner-selection.hits.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='selection-again', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=[], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=False)
2026-10-12 03:23:55,022 [ERROR] opengate_gate_tree: An error occurred: Tree 'Hits' does not hold a branch that identifies a known Hits structure: neither 'volumeID' nor 'volumeID[0]' is present. Branches present: ['eventID', 'edep', 'posX', 'posY', 'posZ'].


exit code: 1


`--skip-hits-validation` extracts it anyway:

In [20]:
!opengate-gate-tree \
    --input-gate-root-file {selection} \
    --output-dir {output} \
    --output-file-title selection-again \
    --gate-tree Hits \
    --output-file-format csv \
    --skip-hits-validation

2026-10-12 03:23:55,854 [INFO] opengate_gate_tree: Run configuration: RunConfig(input_gate_root_file=PosixPath('.tmp/tutorial-04-command-line/scanner-selection.hits.root'), output_dir=PosixPath('.tmp/tutorial-04-command-line'), output_file_title='selection-again', gate_tree=<GateTree.HITS: 'Hits'>, output_file_format=<OutputFileFormat.CSV: 'csv'>, branches_to_extract=[], input_tree_name=None, merge_hits_trees=False, write_statistics=False, skip_hits_validation=True)
2026-10-12 03:23:55,861 [INFO] opengate_gate_tree: Extracted 500 entries and 5 branches from tree 'Hits'.
2026-10-12 03:23:55,866 [INFO] opengate_gate_tree: Done. Output saved to file: .tmp/tutorial-04-command-line/selection-again.hits.csv


The price is visible in the log: there is no line saying which structure the
tree has, because nothing was recognised. A file GATE wrote with a structure the
package does not know is extracted the same way — and nothing then checks that
it holds the branches a "Hits" tree should.

### The Same Run In Python

The command is a thin sequence of calls to the package. The run with
`--branches-to-extract` and `--statistics` above is, in Python:

In [21]:
from pathlib import Path

from opengate_gate_tree import (
    GateTree,
    OutputFileFormat,
    RootFile,
    build_output_file_path,
    build_statistics_file_path,
    compute_statistics,
    write_statistics,
    write_tree,
)

library_output = Path(output) / "from-python"

with RootFile(Path(scanner)) as root_file:
    detection = root_file.detect_hits_tree()
    data = root_file.read(GateTree.HITS, ["runID", "eventID", "trackID", "edep", "processName"])

written = write_tree(
    data,
    build_output_file_path(
        library_output, "scanner-energies", GateTree.HITS, OutputFileFormat.HDF5
    ),
    OutputFileFormat.HDF5,
)
report = write_statistics(
    compute_statistics(data, detection),
    build_statistics_file_path(library_output, "scanner-energies", GateTree.HITS),
)
written.name, report.name

('scanner-energies.hits.hdf5', 'scanner-energies.hits.stats.json')

The two runs wrote the same thing. The data, read back from both HDF5
files, is equal branch by branch, and so are the two reports:

In [22]:
import json

import h5py
import numpy as np

with (
    h5py.File(Path(output) / "scanner-energies.hits.hdf5") as from_command,
    h5py.File(written) as from_python,
):
    same_data = all(
        np.array_equal(from_command["Hits"][name][:], from_python["Hits"][name][:])
        for name in from_command["Hits"]
    )

same_report = json.loads(
    (Path(output) / "scanner-energies.hits.stats.json").read_text()
) == json.loads(report.read_text())

same_data, same_report

(True, True)

### When The Command Is Handier

The command earns its place where Python would be ceremony: many files at once,
from a shell, with nothing to import. In a terminal, extracting the energies of
every ROOT file in a directory is one loop:

```bash
for file in *.root; do
    opengate-gate-tree \
        --input-gate-root-file "$file" \
        --output-dir out \
        --output-file-title "$(basename "$file" .root)" \
        --gate-tree Hits \
        --output-file-format csv \
        --branches-to-extract eventID edep 2>/dev/null \
    || echo "failed: $file"
done
```

In the notebook the loop is written in Python, because IPython reads a `$` in a
`!` line as the start of a Python name. The body is the same command, run for
three example files that hold one "Hits" tree each:

In [23]:
batch = os.path.join(output, "batch")

for name in [
    "hits-a1-no-system.root",
    "hits-a2-cylindrical-pet.root",
    "positronium-source-all-variants.root",
]:
    example = relative(paths.input_example(name))
    title = name.removesuffix(".root")
    !opengate-gate-tree \
        --input-gate-root-file {example} \
        --output-dir {batch} \
        --output-file-title {title} \
        --gate-tree Hits \
        --output-file-format csv \
        --branches-to-extract eventID edep 2>/dev/null \
    || echo "failed: {example}"

In [24]:
sorted(os.listdir(batch))

['hits-a1-no-system.hits.csv',
 'hits-a2-cylindrical-pet.hits.csv',
 'positronium-source-all-variants.hits.csv']

The log is sent away here with `2>/dev/null`, and the exit code alone says
whether a file needs a look; nothing failed. The same holds when the next step
is not Python at all — a C++ program reading the ROOT output, or a colleague
with a spreadsheet and a CSV.

## Summary

`opengate-gate-tree` reads one tree, keeps the branches asked for, writes them
as ROOT, HDF5 or CSV under `<title>.<tree>.<format>`, and with `--statistics`
leaves a report beside them; its exit code says whether it worked. It does
exactly what a few lines of Python do — the choice between the two is about
where the work happens, not about what can be done.

## Bibliography

- [Command line](https://opengate-gate-tree.readthedocs.io/en/latest/guide/cli.html)
  — every option, the exit codes, and what is checked
- [Exporting files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/export.html)
  — how an output file is named and what each format preserves
- [Merging trees](https://opengate-gate-tree.readthedocs.io/en/latest/guide/merging.html)
  — files holding hits in several trees
- [The "Hits" Tree](https://opengate-gate-tree.readthedocs.io/en/latest/guide/hits.html)
  — the structure check, and when to turn it off
- [Statistics](https://opengate-gate-tree.readthedocs.io/en/latest/guide/statistics.html)
  — the report written by `--statistics`
- [Using the package as a library](https://opengate-gate-tree.readthedocs.io/en/latest/guide/library.html)
  — the calls the command is built from

## Changelog

- v1: written.